# 11 STI 插槽（ES3 + A35）：接入 10_sti_garch 的仓位

总仓位 = **90% × 09 的 6336 票投票** + **10% × 新加坡部分**（ES3 比例由 10 号 GARCH 决定，其余持有 A35 债券）。

**默认读取 `10_sti_garch.ipynb` 导出的 `results/signals/sti_weights_history.csv`**（列：`date`, `es3_weight`, `a35_weight`）。文件不存在时，这 10% 放现金，输出里会标注"未接入"；文件里没有 `a35_weight` 列（旧版）时，A35 部分放现金。

## 接口约定
| 项目 | 规则 |
|---|---|
| 输入 `px` | DataFrame，index = 美股交易日；列 `open`、`close`（ES3 美元价）、`local_close`（新元收盘价） |
| 输出 | `pd.Series`，index 与 `px` 相同，值在 0~1（占这 10% 的比例；其余为现金），不能有 NaN |
| 时点 | 第 t 日的值只能用 t 日收盘及以前的数据；t+1 日开盘成交。10 号文件的仓位是新加坡收盘后定的（早于美股当日收盘），用"取不晚于该美股日的最近一个新加坡日"对齐，不会用到未来 |
| 成本 | 不在这里算。全项目统一：ES3、A35 单边各 10 bp（10 号回测与 12 号账本一致） |

**注意**：防偷看检查只对"函数里直接算信号"的写法有意义。读 CSV 时，检查必然通过，因果性要靠 10 号文件自身保证（它的回测用 `T 日收盘预测 → T+1 开盘成交`）。

In [ ]:
import numpy as np
import pandas as pd

SLEEVE_SHARE = 0.10          # 10% 仓位给 ES3 插槽


In [ ]:
# =====================================================================
# 默认：读取 10_sti_garch 导出的每日仓位 CSV。想换成别的策略，只改这一格。
# =====================================================================
import os
from pathlib import Path

def find_weights_csv():
    env = os.environ.get('STI_WEIGHTS_CSV')
    if env and Path(env).is_file(): return Path(env)
    for root in [Path.cwd(), *Path.cwd().parents]:
        p = root / 'results' / 'signals' / 'sti_weights_history.csv'
        if p.is_file(): return p
    return None

def _read_weights(path):
    return pd.read_csv(path, index_col=0, parse_dates=True).sort_index()

def _align(s, px):
    u = s.reindex(s.index.union(px.index)).ffill()               # 取不晚于该日的最近一个新加坡日的仓位
    return u.reindex(px.index).fillna(0.0)

def bond_sleeve_signal(px):
    """这 10% 里持有 A35 的比例（0~1）。10 号未导出 a35_weight 时返回 0（即放现金）。"""
    path = find_weights_csv()
    if path is None:
        return pd.Series(0.0, index=px.index)
    df = _read_weights(path)
    if 'a35_weight' not in df.columns:
        return pd.Series(0.0, index=px.index)
    return _align(df['a35_weight'].dropna().clip(0, 1), px)

def es3_sleeve_signal(px):
    """px: DataFrame(index=美股交易日, columns=['open','close','local_close'])
    返回: Series(index=px.index)，0~1，表示这 10% 里买入 ES3 的比例"""
    path = find_weights_csv()
    if path is None:
        return pd.Series(0.0, index=px.index)                    # 未接入：放现金
    s = _read_weights(path)['es3_weight'].dropna().clip(0, 1)
    return _align(s, px)


In [ ]:
# ================= 以下是框架，不要改 =================
def validate_sleeve(fn, px, n_checks=8, seed=0):
    """检查：格式、范围、无 NaN、无未来数据。"""
    w = fn(px)
    assert isinstance(w, pd.Series), '必须返回 pd.Series'
    assert w.index.equals(px.index), 'index 必须和 px 完全一致'
    assert not w.isna().any(), '不能有 NaN（暂时不开仓请填 0）'
    assert ((w >= 0) & (w <= 1)).all(), '取值必须在 0~1 之间'
    rng = np.random.default_rng(seed)
    cand = np.arange(len(px) // 3, len(px) - 1)
    for k in rng.choice(cand, size=min(n_checks, len(cand)), replace=False):
        part = fn(px.iloc[:k + 1])
        if abs(float(part.iloc[-1]) - float(w.iloc[k])) > 1e-9:
            raise AssertionError(f'疑似使用了未来数据：{px.index[k].date()} 的值，'
                                 f'截断数据下为 {part.iloc[-1]:.4f}，完整数据下为 {w.iloc[k]:.4f}')
    return w

def get_sleeve_weights(px, check=True):
    """09 调用的入口：返回通过检查的 ES3 比例序列（0~1）。"""
    return validate_sleeve(es3_sleeve_signal, px) if check else es3_sleeve_signal(px)

def get_bond_weights(px, check=True):
    """09/12 调用：返回这 10% 里持有 A35 的比例（0~1），并检查 ES3 + A35 ≤ 1。"""
    b = validate_sleeve(bond_sleeve_signal, px) if check else bond_sleeve_signal(px)
    e = es3_sleeve_signal(px)
    assert ((e + b) <= 1 + 1e-9).all(), 'ES3 + A35 的比例不能超过 1'
    return b

def sleeve_is_connected(px):
    return bool(get_sleeve_weights(px, check=False).abs().sum() > 0)

def combine_with_sleeve(weights, sleeve_w, sleeve_share=SLEEVE_SHARE, bond_w=None):
    """weights: 6336 票投票目标权重（列为 TICKS，不含 ES3/A35）；sleeve_w: ES3 比例 0~1；
    bond_w: A35 比例 0~1（None = 不持有 A35，这部分为现金）。
    返回含 ES3、A35 列的目标权重；各行之和 ≤ 1，缺口是现金。"""
    w = weights * (1 - sleeve_share)
    s = sleeve_w.reindex(weights.index).ffill().fillna(0.0).clip(0, 1)
    w['ES3'] = sleeve_share * s
    if bond_w is not None:
        b = bond_w.reindex(weights.index).ffill().fillna(0.0).clip(0, 1)
        w['A35'] = sleeve_share * b
    return w


In [ ]:
# 自测（被 09 加载时跳过本格）
rng = np.random.default_rng(1)
idx = pd.bdate_range('2020-01-01', periods=400)
close = pd.Series(30 * np.exp(np.cumsum(rng.normal(0, 0.01, len(idx)))), index=idx)
px_demo = pd.DataFrame({'open': close.shift(1).fillna(close), 'close': close, 'local_close': close * 1.3})
w = get_sleeve_weights(px_demo)
print('通过检查 | 接入状态:', '已接入' if sleeve_is_connected(px_demo) else '未接入（这10%放现金）')
